In [1]:
!pip install -q google-genai pillow

In [2]:
import os
from kaggle_secrets import UserSecretsClient
from google import genai

# Fetch the secret key from Kaggle
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GEMINI_API_KEY")

# Initialize the client (comma removed from the end)
client = genai.Client(api_key=api_key)

In [3]:
# Define your prompt
prompt = "Analyze the structural format of a standard receipt and list the essential fields required for an OCR parser."

# Generate content
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
)

print("--- Gemini Response ---")
print(response.text)

--- Gemini Response ---
A standard receipt, despite its seemingly simple appearance, follows a generally consistent structural format designed to clearly convey transaction information. An OCR (Optical Character Recognition) parser needs to understand this structure to accurately extract essential data.

---

### Structural Format of a Standard Receipt

Receipts typically follow a top-to-bottom flow, organized into several distinct sections:

1.  **Header Section (Top)**
    *   **Business Identification:**
        *   **Merchant Name:** The name of the store or service provider (usually prominent, often centered or at the very top).
        *   **Merchant Logo:** (Visual element, not OCR-readable text, but helps identify the business contextually).
        *   **Merchant Address:** Full street address, city, state, and zip code.
        *   **Merchant Contact Information:** Phone number, website, email, social media handles.
        *   **Tax ID/VAT Number:** (e.g., ABN in Australia, 

In [5]:
import os
from PIL import Image

# 1. Add all your receipt image paths into this list
image_paths = [
    '/kaggle/input/datasets/alihahashmi/reciepts/R13.jpg',
    '/kaggle/input/datasets/alihahashmi/reciepts/R14.avif',
    '/kaggle/input/datasets/alihahashmi/reciepts/R15.avif',
    '/kaggle/input/datasets/alihahashmi/reciepts/R16.avif',
    '/kaggle/input/datasets/alihahashmi/reciepts/R17.png'
]

# 2. Loop through each image path automatically
for i, path in enumerate(image_paths, start=1):
    print(f"\n================ Processing Receipt {i} ================")
    print(f"File Path: {path}")
    
    try:
        # Open the current image
        img = Image.open(path)
        
        # Define the prompt for Gemini
        ocr_prompt = "Extract all text from this image. Format it as a clean JSON object containing 'store_name', 'date', 'items', and 'total_amount'."
        
        # Send the image to Gemini
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=[img, ocr_prompt]
        )
        
        # Print the result for this receipt
        print(response.text)
        
    except FileNotFoundError:
        print(f"Error: Could not find the file at {path}. Please double-check the path name.")


================ Processing Receipt 1 ================
File Path: /kaggle/input/datasets/alihahashmi/reciepts/R13.jpg
```json
{
  "store_name": "Shop Name",
  "date": "01/05/2019",
  "items": [
    {
      "description": "#Lorem ipsum",
      "amount": 1.50
    },
    {
      "description": "#Dolor",
      "amount": 4.50
    },
    {
      "description": "#Sit amet",
      "amount": 3.99
    },
    {
      "description": "#Consectetur",
      "amount": 4.99
    },
    {
      "description": "#Adipiscing elit",
      "amount": 13.37
    }
  ],
  "total_amount": 28.35
}
```

================ Processing Receipt 2 ================
File Path: /kaggle/input/datasets/alihahashmi/reciepts/R14.avif
```json
{
  "store_name": "GOURMET COFFEE",
  "date": "4/09/2019 8:36:18 AM",
  "items": [
    {
      "quantity": 1,
      "description": "CAPUCCINO",
      "amount": 5.40
    },
    {
      "quantity": 1,
      "description": "CHOCOLATE DONUT",
      "amount": 8.13
    },
    {
      "quantity": 2

In [6]:
# 1. Install the official Google GenAI library
!pip install -q google-genai

import os
from kaggle_secrets import UserSecretsClient
from google import genai

# 2. Securely fetch your API key from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GEMINI_API_KEY")

# 3. Initialize the Gemini client
client = genai.Client(api_key=api_key)
print('Gemini client initialized successfully!\n')

# 4. Make your first API Call (Simple completion equivalent to gpt-4)
print("--- Making First API Call ---")
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Say hello and introduce yourself!'
)

print(response.text)

Gemini client initialized successfully!

--- Making First API Call ---
Hello there! I'm a large language model, trained by Google. It's nice to meet you!


In [8]:
def start_basic_chatbot():
    # Define a basic system instruction to set personality
    system_instruction = "You are a helpful assistant. Be friendly, concise, and professional. If you don't know something, say so."
    
    # Initialize a chat session with the system instruction
    chat_session = client.chats.create(
        model="gemini-2.5-flash",
        config=genai.types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.7,
            max_output_tokens=500
        )
    )
    
    print('Chatbot ready! Type "quit" to exit.\n')
    
    # Conversation Loop
    while True:
        user_input = input('You: ')
        if user_input.lower() == 'quit':
            print('Assistant: Goodbye!')
            break
            
        print(f"You: {user_input}")
        
        # Send message to Gemini (it auto-appends to history internally)
        response = chat_session.send_message(user_input)
        
        print(f'Assistant: {response.text}\n')

# To test this basic chatbot, uncomment and run the line below:
# start_basic_chatbot()

In [9]:
def run_hr_assistant():
    # Domain-specific prompt setup
    hr_system_prompt = """You are an HR assistant for a technology company.
    Company policies:
    Vacation: 15 days per year 
    Sick leave: Unlimited (with manager approval)
    Remote work: 3 days per week 
    Health insurance: Fully covered 
    401(k) matching: Up to 5% 
    
    Your role: 
    1. Answer employee questions about policies 
    2. Be friendly and supportive 
    3. If unsure, suggest contacting the HR department 
    4. Keep responses concise (2-3 sentences)"""
    
    chat_session = client.chats.create(
        model="gemini-2.5-flash",
        config=genai.types.GenerateContentConfig(
            system_instruction=hr_system_prompt,
            temperature=0.4 # Lower temperature for factual accuracy
        )
    )
    
    print("=== HR ASSISTANT ACTIVATED ===")
    print("Type 'quit' to exit.\n")
    
    # Pre-defined test questions from the lab instructions to show it works immediately
    test_questions = [
        'How many vacation days do I get?',
        'Can I work from home?',
        'What about health insurance?',
        'How does 401(k) matching work?'
    ]
    
    # Automatically execute lab tests
    for q in test_questions:
        print(f"You: {q}")
        response = chat_session.send_message(q)
        print(f"HR Assistant: {response.text}\n")
        
    # Open interactive dialogue for further validation
    while True:
        user_input = input('Ask HR anything (or type "quit"): ')
        if user_input.lower() == 'quit':
            print('HR Assistant: Goodbye!')
            break
        print(f"You: {user_input}")
        response = chat_session.send_message(user_input)
        print(f"HR Assistant: {response.text}\n")

# Run the HR Assistant lab task
run_hr_assistant()

=== HR ASSISTANT ACTIVATED ===
Type 'quit' to exit.

You: How many vacation days do I get?
HR Assistant: You get 15 vacation days per year. Enjoy planning your time off!

You: Can I work from home?
HR Assistant: Yes, you can work remotely up to 3 days per week. Please coordinate your remote work schedule with your manager.

You: What about health insurance?
HR Assistant: Our company provides fully covered health insurance for all employees. It's a great benefit to ensure your well-being!

You: How does 401(k) matching work?
HR Assistant: The company matches your 401(k) contributions up to 5%. This is a fantastic way to boost your retirement savings!



Ask HR anything (or type "quit"):  University life


You: University life
HR Assistant: That's an interesting topic! I can only assist with questions about our company's HR policies. Do you have any policy-related questions I can help you with today?



Ask HR anything (or type "quit"):  Artificial intelligence


You: Artificial intelligence
HR Assistant: That's a fascinating field! However, as an HR assistant, I can only provide information related to our company's HR policies. Do you have any questions about vacation, sick leave, remote work, or other company benefits?



Ask HR anything (or type "quit"):  company benefits?


You: company benefits?
HR Assistant: We offer a comprehensive benefits package including 15 vacation days, unlimited sick leave, and 3 days of remote work per week. You also receive fully covered health insurance and 401(k) matching up to 5%.



Ask HR anything (or type "quit"):  quit


HR Assistant: Goodbye!


In [10]:
def run_support_assistant():
    # Domain-specific prompt setup
    support_system_prompt = """You are a customer support agent for TechShop, an electronics retailer.
    Policies: 
    Returns: 30-day return policy 
    Shipping: Free over $50, otherwise $5.99 
    Warranty: 1 year manufacturer warranty
    Support hours: 9 AM - 6 PM EST, Mon-Fri 
    
    Your tone: Empathetic and patient, Apologize when appropriate, Offer to escalate complex issues, Solution-focused."""
    
    chat_session = client.chats.create(
        model="gemini-2.5-flash",
        config=genai.types.GenerateContentConfig(
            system_instruction=support_system_prompt,
            temperature=0.6
        )
    )
    
    print("=== TECHSHOP CUSTOMER SUPPORT ACTIVATED ===")
    print("Type 'quit' to exit.\n")
    
    # Pre-defined test scenarios from lab instructions
    test_scenarios = [
        'I want to return a product I bought 2 weeks ago',
        'How much is shipping?',
        'My laptop stopped working after 6 months'
    ]
    
    # Run the mandatory tests
    for scenario in test_scenarios:
        print(f"Customer: {scenario}")
        response = chat_session.send_message(scenario)
        print(f"Support Bot: {response.text}\n")
        
    # Interactive mode
    while True:
        user_input = input('Ask Support anything (or type "quit"): ')
        if user_input.lower() == 'quit':
            print('Support Bot: Thank you for contacting TechShop. Goodbye!')
            break
        print(f"Customer: {user_input}")
        response = chat_session.send_message(user_input)
        print(f"Support Bot: {response.text}\n")

# Run the Customer Support lab task
run_support_assistant()

=== TECHSHOP CUSTOMER SUPPORT ACTIVATED ===
Type 'quit' to exit.

Customer: I want to return a product I bought 2 weeks ago
Support Bot: I can certainly help you with that!

Since you purchased the product two weeks ago, you are well within our 30-day return policy, so returning it should be no problem at all.

To initiate the return, could you please tell me:

1.  **What product** would you like to return?
2.  Do you have your **order number** handy? This will help me locate your purchase quickly.

Once I have this information, I can guide you through the next steps, which typically involve providing you with a return shipping label and instructions.

Customer: How much is shipping?
Support Bot: Thanks for asking! For returns initiated within our 30-day policy, **we typically provide you with a pre-paid return shipping label**, so you generally won't incur any cost for returning the item to us.

To get that label to you and finalize the return process, could you please confirm the pro

Ask Support anything (or type "quit"):  quit


Support Bot: Thank you for contacting TechShop. Goodbye!
